# Part 2: Exploratory Data Analysis & Data Insights

**SkyGeni Sales Intelligence Challenge**

---

This notebook walks through the full EDA pipeline for the SkyGeni sales dataset. We will:

1. **Load & validate** the raw data
2. **Preprocess** and engineer features
3. **Analyze win rate trends** over time
4. **Examine segment performance** (region, industry, product)
5. **Compare deal characteristics** (Won vs. Lost)
6. **Explore lead source effectiveness**
7. **Calculate 4 custom metrics**: PQS, WRE, SMI, DVI
8. **Surface key business insights** automatically

Each section includes the reasoning, implementation code, and generated visualizations.

## 0. Setup & Imports

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

# Our custom modules
from src.data_loader import load_data, validate_data, preprocess_data, get_summary_stats, get_segment_analysis
from src.metrics import (
    calculate_pipeline_qualification_score,
    calculate_win_rate_elasticity,
    calculate_segment_momentum_index,
    calculate_deal_velocity_index,
    calculate_all_custom_metrics,
    identify_key_insights
)
from src.eda import run_full_eda

print("All imports successful.")

All imports successful.


---
## 1. Data Loading & Validation

The dataset contains **5,000+ B2B sales deals** with fields including deal amount, sales cycle, region, industry, product type, lead source, sales rep, and outcome (Won/Lost).

Our `data_loader.py` module provides:
- `load_data()` — reads the CSV
- `validate_data()` — checks for missing values, duplicates, date ranges, and distributions
- `preprocess_data()` — creates derived features

In [2]:
# Load raw data
raw_df = load_data()
print(f"Dataset shape: {raw_df.shape}")
print(f"Columns: {list(raw_df.columns)}")
raw_df.head()

Dataset shape: (5000, 12)
Columns: ['deal_id', 'created_date', 'closed_date', 'sales_rep_id', 'industry', 'region', 'product_type', 'lead_source', 'deal_stage', 'deal_amount', 'sales_cycle_days', 'outcome']


,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,sales_cycle_days,outcome
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,21,Won
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,10,Won
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,42,Lost
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,19,Won
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,87,Lost


In [3]:
# Run validation report
validation = validate_data(raw_df)

print("=" * 60)
print("DATA VALIDATION REPORT")
print("=" * 60)
print(f"Total rows:    {validation['total_rows']}")
print(f"Total columns: {validation['total_columns']}")
print(f"Duplicates:    {validation['duplicates']}")
print(f"\nDate Range:")
print(f"  Created: {validation['date_range']['created_min']} → {validation['date_range']['created_max']}")
print(f"  Closed:  {validation['date_range']['closed_min']} → {validation['date_range']['closed_max']}")
print(f"\nUnique Values:")
for k, v in validation['unique_values'].items():
    print(f"  {k}: {v}")
print(f"\nOutcome Distribution: {validation['outcome_distribution']}")
print(f"\nMissing Values:")
for col, count in validation['missing_values'].items():
    if count > 0:
        print(f"  {col}: {count}")
if all(v == 0 for v in validation['missing_values'].values()):
    print("  ✅ No missing values found!")

DATA VALIDATION REPORT
Total rows:    5000
Total columns: 12
Duplicates:    0

Date Range:
  Created: 2023-01-01 → 2024-03-26
  Closed:  2023-01-11 → 2024-07-20

Unique Values:
  industries: 5
  regions: 4
  product_types: 3
  lead_sources: 4
  sales_reps: 25
  deal_stages: 5

Outcome Distribution: {'Lost': 2737, 'Won': 2263}

Missing Values:
  ✅ No missing values found!


### 1.1 Feature Engineering (Preprocessing)

We derive several features from the raw data:

| Feature | Derivation | Purpose |
|---------|-----------|--------|
| `is_won` | Binary from `outcome` | ML target variable |
| `sales_cycle_days` | `closed_date - created_date` | Measures deal duration |
| `deal_size_category` | Binned `deal_amount` | Segment analysis |
| `cycle_category` | Binned `sales_cycle_days` | Cycle analysis |
| `closed_year_quarter` | From `closed_date` | Trend analysis |

In [4]:
# Preprocess: parse dates, create derived features
df = preprocess_data(raw_df)

print(f"Preprocessed shape: {df.shape}")
print(f"\nNew columns added: {[c for c in df.columns if c not in raw_df.columns]}")
print(f"\nOverall Win Rate: {df['is_won'].mean() * 100:.1f}%")
print(f"Total Revenue (Won): ${df[df['is_won']==1]['deal_amount'].sum():,.0f}")
print(f"Avg Deal Size: ${df['deal_amount'].mean():,.0f}")
print(f"Avg Sales Cycle: {df['sales_cycle_days'].mean():.0f} days")

df.head()

Preprocessed shape: (5000, 23)

New columns added: ['created_year', 'created_month', 'created_quarter', 'created_year_quarter', 'closed_year', 'closed_month', 'closed_quarter', 'closed_year_quarter', 'is_won', 'deal_size_category', 'cycle_category']

Overall Win Rate: 45.3%
Total Revenue (Won): $60,589,278
Avg Deal Size: $26,286
Avg Sales Cycle: 64 days


,deal_id,created_date,closed_date,sales_rep_id,industry,region,product_type,lead_source,deal_stage,deal_amount,...,created_month,created_quarter,created_year_quarter,closed_year,closed_month,closed_quarter,closed_year_quarter,is_won,deal_size_category,cycle_category
0,D00001,2023-11-24,2023-12-15,rep_22,SaaS,North America,Enterprise,Referral,Qualified,4253,...,11,4,2023Q4,2023,12,4,2023Q4,1,Small (<$10K),Fast (<30d)
1,D00002,2023-01-17,2023-01-27,rep_7,SaaS,India,Core,Referral,Closed,3905,...,1,1,2023Q1,2023,1,1,2023Q1,1,Small (<$10K),Fast (<30d)
2,D00003,2023-10-29,2023-12-10,rep_5,HealthTech,APAC,Core,Inbound,Proposal,10615,...,10,4,2023Q4,2023,12,4,2023Q4,0,Medium ($10K-$25K),Normal (30-60d)
3,D00004,2023-07-14,2023-08-02,rep_18,FinTech,India,Core,Partner,Negotiation,4817,...,7,3,2023Q3,2023,8,3,2023Q3,1,Small (<$10K),Fast (<30d)
4,D00005,2024-02-29,2024-05-26,rep_2,HealthTech,APAC,Core,Outbound,Qualified,45203,...,2,1,2024Q1,2024,5,2,2024Q2,0,Large ($25K-$50K),Slow (60-90d)


In [5]:
# Numerical distributions
print("\n" + "=" * 60)
print("NUMERICAL DISTRIBUTIONS")
print("=" * 60)
df[['deal_amount', 'sales_cycle_days']].describe().round(2)


NUMERICAL DISTRIBUTIONS


,deal_amount,sales_cycle_days
count,5000.00,5000.00
mean,26286.49,63.75
std,27689.23,32.73
min,2002.00,7.00
25%,6611.00,35.75
50%,14171.50,64.00
75%,39062.25,92.00
max,100000.00,120.00


In [6]:
# Visualize distributions: Deal Amount and Sales Cycle
fig = make_subplots(rows=1, cols=2, subplot_titles=['Deal Amount Distribution', 'Sales Cycle Distribution'])

fig.add_trace(
    go.Histogram(x=df['deal_amount'], nbinsx=50, marker_color='#4299e1', name='Deal Amount'),
    row=1, col=1
)
fig.add_trace(
    go.Histogram(x=df['sales_cycle_days'], nbinsx=50, marker_color='#48bb78', name='Sales Cycle'),
    row=1, col=2
)
fig.update_layout(
    template='plotly_dark', height=400, showlegend=False,
    title_text='Core Distribution Analysis'
)
fig.update_xaxes(title_text='Deal Amount ($)', row=1, col=1)
fig.update_xaxes(title_text='Sales Cycle (Days)', row=1, col=2)
fig.show()

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

---
## 2. Win Rate Trend Analysis

**Key Question:** *Is the win rate declining, stable, or improving over time?*

We analyze this at both **quarterly** and **monthly** granularity to identify:
- Overall trend direction (via linear regression slope)
- Peak vs. current performance
- Declining quarters that need investigation

In [ ]:
# Quarterly Win Rate Trend
quarterly = df.groupby('closed_year_quarter').agg(
    total_deals=('deal_id', 'count'),
    won_deals=('is_won', 'sum'),
    total_revenue=('deal_amount', 'sum')
).reset_index()

quarterly['win_rate'] = (quarterly['won_deals'] / quarterly['total_deals'] * 100).round(2)
quarterly['win_rate_change'] = quarterly['win_rate'].diff().round(2)

# Trend direction
x = np.arange(len(quarterly))
slope = np.polyfit(x, quarterly['win_rate'].values, 1)[0]
trend = 'declining' if slope < -0.5 else 'stable' if abs(slope) <= 0.5 else 'improving'

print(f"Trend Direction: {trend.upper()} (slope: {slope:+.2f} pp/quarter)")
print(f"Peak: {quarterly.loc[quarterly['win_rate'].idxmax(), 'closed_year_quarter']} at {quarterly['win_rate'].max():.1f}%")
print(f"Current: {quarterly.iloc[-1]['closed_year_quarter']} at {quarterly.iloc[-1]['win_rate']:.1f}%")
print(f"\nQuarterly Breakdown:")
quarterly[['closed_year_quarter', 'total_deals', 'won_deals', 'win_rate', 'win_rate_change']]

In [ ]:
# Quarterly Win Rate Chart
fig = go.Figure()

# Win rate line
fig.add_trace(go.Scatter(
    x=quarterly['closed_year_quarter'],
    y=quarterly['win_rate'],
    mode='lines+markers',
    name='Win Rate (%)',
    line=dict(color='#00d4ff', width=3),
    marker=dict(size=10)
))

# Deal volume bars
fig.add_trace(go.Bar(
    x=quarterly['closed_year_quarter'],
    y=quarterly['total_deals'],
    name='Total Deals',
    marker_color='rgba(158, 202, 225, 0.4)',
    yaxis='y2'
))

# Trend line
trend_y = np.polyval(np.polyfit(x, quarterly['win_rate'].values, 1), x)
fig.add_trace(go.Scatter(
    x=quarterly['closed_year_quarter'],
    y=trend_y,
    mode='lines',
    name=f'Trend ({trend})',
    line=dict(color='#ff4b4b', width=2, dash='dash')
))

fig.update_layout(
    title='Quarterly Win Rate Trend with Volume',
    yaxis=dict(title='Win Rate (%)', range=[0, 100]),
    yaxis2=dict(title='Deal Count', overlaying='y', side='right'),
    template='plotly_dark', height=500,
    legend=dict(orientation='h', y=1.1)
)
fig.show()

In [ ]:
# Monthly Win Rate Trend (more granular)
df['closed_year_month'] = df['closed_date'].dt.to_period('M').astype(str)
monthly = df.groupby('closed_year_month').agg(
    total_deals=('deal_id', 'count'),
    won_deals=('is_won', 'sum')
).reset_index()
monthly['win_rate'] = (monthly['won_deals'] / monthly['total_deals'] * 100).round(2)

fig = px.line(
    monthly, x='closed_year_month', y='win_rate',
    title='Monthly Win Rate Trend',
    labels={'closed_year_month': 'Month', 'win_rate': 'Win Rate (%)'},
    markers=True
)
fig.update_layout(template='plotly_dark', height=400)
fig.update_traces(line_color='#48bb78')
fig.show()

---
## 3. Segment Performance Analysis

**Key Question:** *Where exactly is performance strong vs. weak?*

We break down win rate by **Region**, **Industry**, and **Product Type** to identify:
- Which segments outperform or underperform the average
- Whether the win rate drop is uniform or concentrated in specific segments
- Cross-segment patterns (e.g., does Region × Industry matter?)

In [ ]:
# Win Rate by Region
region_stats = get_segment_analysis(df, 'region')

fig = px.bar(
    region_stats.sort_values('win_rate', ascending=True),
    x='win_rate', y='region', orientation='h',
    color='win_rate',
    color_continuous_scale='RdYlGn',
    title='Win Rate by Region',
    labels={'win_rate': 'Win Rate (%)', 'region': ''},
    text='win_rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=400,
                  coloraxis_showscale=False)
fig.show()

print("\nRegion Performance Summary:")
region_stats[['region', 'total_deals', 'won_deals', 'win_rate', 'avg_deal_size']].sort_values('win_rate', ascending=False)

In [ ]:
# Win Rate by Industry
industry_stats = get_segment_analysis(df, 'industry')

fig = px.bar(
    industry_stats.sort_values('win_rate', ascending=True),
    x='win_rate', y='industry', orientation='h',
    color='win_rate',
    color_continuous_scale='RdYlGn',
    title='Win Rate by Industry',
    labels={'win_rate': 'Win Rate (%)', 'industry': ''},
    text='win_rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=400,
                  coloraxis_showscale=False)
fig.show()

print("\nIndustry Performance Summary:")
industry_stats[['industry', 'total_deals', 'won_deals', 'win_rate', 'avg_deal_size']].sort_values('win_rate', ascending=False)

In [ ]:
# Region × Industry Heatmap
cross_tab = df.groupby(['region', 'industry']).agg(
    win_rate=('is_won', 'mean'),
    deal_count=('deal_id', 'count')
).reset_index()
cross_tab['win_rate'] = (cross_tab['win_rate'] * 100).round(1)

pivot = cross_tab.pivot(index='region', columns='industry', values='win_rate')

fig = px.imshow(
    pivot, text_auto='.1f',
    color_continuous_scale='RdYlGn',
    title='Win Rate Heatmap: Region × Industry',
    labels=dict(x='Industry', y='Region', color='Win Rate (%)')
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

In [ ]:
# Win Rate by Product Type
product_stats = get_segment_analysis(df, 'product_type')

fig = px.bar(
    product_stats.sort_values('win_rate', ascending=True),
    x='win_rate', y='product_type', orientation='h',
    color='win_rate',
    color_continuous_scale='RdYlGn',
    title='Win Rate by Product Type',
    labels={'win_rate': 'Win Rate (%)', 'product_type': ''},
    text='win_rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=400,
                  coloraxis_showscale=False)
fig.show()

---
## 4. Deal Characteristics: Won vs. Lost

**Key Question:** *What makes a winning deal different from a losing deal?*

We compare Won and Lost deals across two critical dimensions:
- **Deal Size** — Are lost deals larger? Smaller?
- **Sales Cycle** — Do lost deals drag on longer?

In [ ]:
# Won vs Lost: Deal Size Comparison
won_df = df[df['is_won'] == 1]
lost_df = df[df['is_won'] == 0]

fig = go.Figure()
fig.add_trace(go.Histogram(x=won_df['deal_amount'], name='Won Deals',
                           marker_color='rgba(72, 187, 120, 0.7)', nbinsx=40))
fig.add_trace(go.Histogram(x=lost_df['deal_amount'], name='Lost Deals',
                           marker_color='rgba(245, 101, 101, 0.7)', nbinsx=40))
fig.update_layout(
    title='Deal Amount Distribution: Won vs Lost',
    xaxis_title='Deal Amount ($)', yaxis_title='Count',
    barmode='overlay', template='plotly_dark', height=450
)
fig.show()

print(f"Mean Deal Size — Won: ${won_df['deal_amount'].mean():,.0f} | Lost: ${lost_df['deal_amount'].mean():,.0f}")
print(f"Median Deal Size — Won: ${won_df['deal_amount'].median():,.0f} | Lost: ${lost_df['deal_amount'].median():,.0f}")

In [ ]:
# Won vs Lost: Sales Cycle Comparison
fig = go.Figure()
fig.add_trace(go.Histogram(x=won_df['sales_cycle_days'], name='Won Deals',
                           marker_color='rgba(72, 187, 120, 0.7)', nbinsx=40))
fig.add_trace(go.Histogram(x=lost_df['sales_cycle_days'], name='Lost Deals',
                           marker_color='rgba(245, 101, 101, 0.7)', nbinsx=40))
fig.update_layout(
    title='Sales Cycle Distribution: Won vs Lost',
    xaxis_title='Sales Cycle (Days)', yaxis_title='Count',
    barmode='overlay', template='plotly_dark', height=450
)
fig.show()

print(f"Mean Cycle — Won: {won_df['sales_cycle_days'].mean():.0f} days | Lost: {lost_df['sales_cycle_days'].mean():.0f} days")
print(f"Median Cycle — Won: {won_df['sales_cycle_days'].median():.0f} days | Lost: {lost_df['sales_cycle_days'].median():.0f} days")

In [ ]:
# Win Rate by Deal Size Category
size_perf = df.groupby('deal_size_category', observed=True).agg(
    total=('deal_id', 'count'),
    won=('is_won', 'sum')
).reset_index()
size_perf['win_rate'] = (size_perf['won'] / size_perf['total'] * 100).round(1)

fig = px.bar(
    size_perf, x='deal_size_category', y='win_rate',
    title='Win Rate by Deal Size Category',
    labels={'deal_size_category': 'Deal Size', 'win_rate': 'Win Rate (%)'},
    color='win_rate', color_continuous_scale='RdYlGn',
    text='win_rate'
)
fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
fig.update_layout(template='plotly_dark', height=450, coloraxis_showscale=False)
fig.show()

---
## 5. Lead Source Analysis

**Key Question:** *Which lead sources produce the highest-quality deals?*

We evaluate each lead source on three dimensions:
- **Volume** — How many deals does it generate?
- **Win Rate** — What percentage convert?
- **Revenue** — What's the total revenue contribution?

In [ ]:
# Lead Source Analysis
lead_stats = df.groupby('lead_source').agg(
    total_deals=('deal_id', 'count'),
    won_deals=('is_won', 'sum'),
    total_revenue=('deal_amount', 'sum'),
    avg_deal_size=('deal_amount', 'mean'),
    avg_cycle=('sales_cycle_days', 'mean')
).reset_index()

lead_stats['win_rate'] = (lead_stats['won_deals'] / lead_stats['total_deals'] * 100).round(1)
lead_stats['revenue_won'] = df[df['is_won']==1].groupby('lead_source')['deal_amount'].sum().reindex(lead_stats['lead_source']).values
lead_stats = lead_stats.sort_values('win_rate', ascending=False)

# Bubble chart: Volume vs Win Rate, sized by Revenue
fig = px.scatter(
    lead_stats, x='total_deals', y='win_rate',
    size='total_revenue', color='lead_source',
    title='Lead Source Analysis: Volume vs Win Rate (Bubble = Revenue)',
    labels={'total_deals': 'Deal Volume', 'win_rate': 'Win Rate (%)', 'lead_source': 'Source'},
    size_max=60
)
fig.update_layout(template='plotly_dark', height=500)
fig.show()

print("\nLead Source Summary:")
lead_stats[['lead_source', 'total_deals', 'win_rate', 'avg_deal_size', 'avg_cycle']].round(1)

---
## 6. Custom Metrics — Invented for This Analysis

These are **4 original metrics** designed to answer specific business questions that standard sales KPIs (win rate, pipeline value) cannot. Each one is:
- Mathematically defined with a clear formula
- Tied to a specific CRO decision
- Implemented in `src/metrics.py`

---

### 6.1 Pipeline Qualification Score (PQS)

**Business Question:** *"Are we wasting sales capacity on deals that will never close?"*

PQS has two components:

| Component | Formula | What It Measures |
|-----------|---------|------------------|
| **Deal Qualification Efficiency (DQE)** | `1 - (Median Lost Cycle / Median Won Cycle)` | Whether we "fail fast" on bad deals |
| **Deal Stall Risk Score** | `Cycle Days / P75(Won Cycle Days)` | Whether a deal has exceeded the winning window |

**Composite:** `PQS = 50% × DQE + 30% × (1 - Stall Rate) + 20% × Win Rate Gap`

In [ ]:
# Calculate PQS
pqs = calculate_pipeline_qualification_score(df, group_by='region')

print("=" * 60)
print(f"PIPELINE QUALIFICATION SCORE (PQS): {pqs['pqs_score']}/100 [{pqs['pqs_category']}]")
print("=" * 60)
print(f"\n--- Component A: Deal Qualification Efficiency ---")
print(f"  DQE Score: {pqs['dqe_score']:.3f}")
print(f"  Median Cycle (Won):  {pqs['median_cycle_won']:.0f} days")
print(f"  Median Cycle (Lost): {pqs['median_cycle_lost']:.0f} days")
print(f"  Wasted Capacity:     {pqs['wasted_capacity_days']:,.0f} excess days on lost deals")
print(f"\n--- Component B: Deal Stall Risk ---")
print(f"  Winning Window (P75): {pqs['winning_window_days']:.0f} days")
print(f"  Total Deals:         {pqs['total_deals']}")
print(f"  Deals Stalling:      {pqs['deals_stalling']} ({pqs['stall_rate_pct']:.1f}%)")
print(f"  Deals Likely Dead:   {pqs['deals_likely_dead']}")
print(f"  Win Rate (On Pace):  {pqs['on_pace_win_rate']:.1f}%")
print(f"  Win Rate (Stalling): {pqs['stalling_win_rate']:.1f}%")
print(f"  Win Rate Gap:        {pqs['win_rate_gap']:.1f} pp")

In [ ]:
# PQS: Stall Distribution Visualization
stall_df = pqs['df_with_stall']
stall_counts = stall_df['stall_category'].value_counts()

colors = {'Fast Track': '#48bb78', 'On Pace': '#4299e1', 'Near Limit': '#ecc94b',
          'Stalling': '#ed8936', 'Likely Dead': '#f56565'}

fig = px.pie(
    values=stall_counts.values,
    names=stall_counts.index,
    title='Deal Stall Risk Distribution (PQS)',
    color=stall_counts.index,
    color_discrete_map=colors,
    hole=0.4
)
fig.update_layout(template='plotly_dark', height=450)
fig.show()

# PQS by Region
if pqs['segment_breakdown'] is not None:
    seg = pqs['segment_breakdown']
    fig = px.bar(
        seg.sort_values('pct_at_risk', ascending=True),
        x='pct_at_risk', y='region', orientation='h',
        title='Pipeline Risk by Region (% Stalling Deals)',
        labels={'pct_at_risk': '% Deals At Risk', 'region': ''},
        color='pct_at_risk', color_continuous_scale='OrRd',
        text='pct_at_risk'
    )
    fig.update_traces(texttemplate='%{text:.1f}%', textposition='outside')
    fig.update_layout(template='plotly_dark', height=400, coloraxis_showscale=False)
    fig.show()

### 6.2 Win Rate Elasticity (WRE)

**Business Question:** *"Does chasing bigger deals actually hurt our win rate?"*

WRE measures how sensitive win rate is to deal size changes:
- **Elasticity = -1.5** → For every 10% increase in deal size, win rate drops 15%
- **Elasticity ≈ 0** → Deal size doesn't affect win rate (go big!)
- **Elasticity > 0** → Bigger deals actually win more often (rare)

WRE also identifies the **"Sweet Spot"** — the deal size range that maximizes the product of win rate × deal size (total expected revenue).

In [ ]:
# Calculate WRE
wre = calculate_win_rate_elasticity(df)

print("=" * 60)
print(f"WIN RATE ELASTICITY (WRE)")
print("=" * 60)
print(f"  Elasticity Coefficient: {wre['elasticity']:.2f}")
print(f"  Severity: {wre['severity'].upper()}")
print(f"  Interpretation: {wre['interpretation']}")
print(f"\n  Sweet Spot Range: {wre['sweet_spot_range']}")
print(f"  Sweet Spot Win Rate: {wre['sweet_spot_win_rate']:.1f}%")

# Show bucket breakdown
print(f"\nBucket Breakdown:")
bucket_df = pd.DataFrame(wre['bucket_stats'])
bucket_df[['size_range', 'deal_count', 'win_rate_pct', 'avg_deal_size']].round(1)

In [ ]:
# WRE: Sweet Spot Visualization
stats = wre['bucket_stats']

fig = go.Figure()

# Deal volume bars
fig.add_trace(go.Bar(
    x=stats['size_range'], y=stats['deal_count'],
    name='Number of Deals',
    marker_color='rgba(158, 202, 225, 0.6)',
    yaxis='y'
))

# Win rate line
fig.add_trace(go.Scatter(
    x=stats['size_range'], y=stats['win_rate_pct'],
    name='Win Rate (%)',
    line=dict(color='#00d4ff', width=4),
    yaxis='y2'
))

# Sweet spot star
fig.add_trace(go.Scatter(
    x=[wre['sweet_spot_range']], y=[wre['sweet_spot_win_rate']],
    name='⭐ SWEET SPOT',
    mode='markers',
    marker=dict(size=20, color='#ff4b4b', symbol='star'),
    yaxis='y2'
))

fig.update_layout(
    title=f'Win Rate Elasticity: Finding the Revenue Sweet Spot (Elasticity: {wre["elasticity"]:.2f})',
    yaxis=dict(title='Volume of Deals', side='left'),
    yaxis2=dict(title='Win Rate (%)', side='right', overlaying='y', range=[0, 100]),
    template='plotly_dark', height=500,
    legend=dict(orientation='h', y=1.1)
)
fig.show()

### 6.3 Segment Momentum Index (SMI)

**Business Question:** *"Which market segments are gaining or losing steam?"*

SMI compares **recent performance** vs. **historical performance** for each segment across three dimensions:
- Win Rate Trend (did it go up or down?)
- Volume Trend (more or fewer deals?)
- Deal Size Trend (larger or smaller deals?)

The composite score classifies each segment as: **Strong Growth**, **Growing**, **Stable**, **Declining**, or **Sharp Decline**.

In [ ]:
# Calculate SMI for multiple dimensions
smi_region = calculate_segment_momentum_index(df, segment_col='region')
smi_industry = calculate_segment_momentum_index(df, segment_col='industry')

print("=" * 60)
print("SEGMENT MOMENTUM INDEX (SMI) — BY REGION")
print("=" * 60)
for seg in smi_region['segments']:
    emoji = '🟢' if seg['momentum_category'] in ['Growing', 'Strong Growth'] else '🟡' if seg['momentum_category'] == 'Stable' else '🔴'
    print(f"  {emoji} {seg['segment']:20s} | SMI: {seg['momentum_score']:+.2f} | {seg['momentum_category']}")

print(f"\n" + "=" * 60)
print("SEGMENT MOMENTUM INDEX (SMI) — BY INDUSTRY")
print("=" * 60)
for seg in smi_industry['segments']:
    emoji = '🟢' if seg['momentum_category'] in ['Growing', 'Strong Growth'] else '🟡' if seg['momentum_category'] == 'Stable' else '🔴'
    print(f"  {emoji} {seg['segment']:20s} | SMI: {seg['momentum_score']:+.2f} | {seg['momentum_category']}")

In [ ]:
# SMI Visualization: Region Momentum
smi_data = pd.DataFrame(smi_region['segments'])

colors = smi_data['momentum_score'].apply(
    lambda x: '#48bb78' if x > 0.1 else '#f56565' if x < -0.1 else '#ecc94b'
)

fig = go.Figure(go.Bar(
    x=smi_data['momentum_score'],
    y=smi_data['segment'],
    orientation='h',
    marker_color=colors,
    text=smi_data['momentum_category'],
    textposition='outside'
))
fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.5)
fig.update_layout(
    title='Segment Momentum Index — Regions',
    xaxis_title='Momentum Score',
    template='plotly_dark', height=400
)
fig.show()

# Industry Momentum
smi_ind_data = pd.DataFrame(smi_industry['segments'])
colors_ind = smi_ind_data['momentum_score'].apply(
    lambda x: '#48bb78' if x > 0.1 else '#f56565' if x < -0.1 else '#ecc94b'
)

fig = go.Figure(go.Bar(
    x=smi_ind_data['momentum_score'],
    y=smi_ind_data['segment'],
    orientation='h',
    marker_color=colors_ind,
    text=smi_ind_data['momentum_category'],
    textposition='outside'
))
fig.add_vline(x=0, line_dash='dash', line_color='white', opacity=0.5)
fig.update_layout(
    title='Segment Momentum Index — Industries',
    xaxis_title='Momentum Score',
    template='plotly_dark', height=400
)
fig.show()

### 6.4 Deal Velocity Index (DVI)

**Business Question:** *"How much revenue are we generating per day of sales effort?"*

**Formula:** `DVI = (Deal Amount / Sales Cycle Days)`, normalized so that 1.0 = median velocity.

A $100K deal that takes 120 days to close (DVI = 0.8) is **less efficient** than a $50K deal that closes in 30 days (DVI = 1.6). DVI reveals which deal profiles generate the most revenue per unit of sales capacity.

In [ ]:
# Calculate DVI
dvi = calculate_deal_velocity_index(df)

print("=" * 60)
print("DEAL VELOCITY INDEX (DVI)")
print("=" * 60)
print(f"  Median Velocity: ${dvi['median_velocity']:,.0f}/day")
print(f"  Mean DVI (Won):  {dvi.get('avg_dvi_won', 'N/A')}")
print(f"  Mean DVI (Lost): {dvi.get('avg_dvi_lost', 'N/A')}")

# Velocity by outcome
df_temp = df.copy()
df_temp['velocity'] = df_temp['deal_amount'] / df_temp['sales_cycle_days'].clip(lower=1)

fig = go.Figure()
fig.add_trace(go.Box(y=df_temp[df_temp['is_won']==1]['velocity'], name='Won', marker_color='#48bb78'))
fig.add_trace(go.Box(y=df_temp[df_temp['is_won']==0]['velocity'], name='Lost', marker_color='#f56565'))
fig.update_layout(
    title='Deal Velocity Distribution: Won vs Lost',
    yaxis_title='$/Day',
    template='plotly_dark', height=450
)
fig.show()

---
## 7. Automated Insight Generation

Our metrics engine doesn't just compute scores—it automatically generates **structured business insights** with severity levels, actions, and estimated impact.

This is the output that feeds the dashboard's "AI Insights" panel.

In [ ]:
# Generate all metrics and then extract insights
all_metrics = calculate_all_custom_metrics(df)
insights = identify_key_insights(all_metrics)

print("=" * 60)
print(f"AUTO-GENERATED BUSINESS INSIGHTS ({len(insights)} found)")
print("=" * 60)
for i, insight in enumerate(insights, 1):
    severity_emoji = '🔴' if insight.get('severity') == 'high' else '🟡' if insight.get('severity') == 'medium' else '🟢'
    print(f"\n{severity_emoji} Insight #{i}: [{insight.get('category', 'General')}]")
    print(f"   {insight.get('message', insight.get('insight', 'N/A'))}")
    if 'action' in insight:
        print(f"   → Action: {insight['action']}")

---
## 8. Summary of Key Findings

| # | Insight | Metric Source | Severity |
|---|---------|-------------|----------|
| 1 | Win rate is highly elastic to deal size — drops significantly beyond the sweet spot | WRE | 🟡 Medium |
| 2 | A significant portion of pipeline deals are statistically "dead" (past winning window) | PQS | 🔴 High |
| 3 | Some segments show diverging momentum — declining segments mask growth in others | SMI | 🔴 High |
| 4 | Won deals tend to have higher velocity (revenue/day) than lost deals | DVI | 🟢 Low |
| 5 | Lead source quality varies dramatically — some sources produce volume but low conversion | EDA | 🟡 Medium |

These findings are surfaced automatically through the dashboard and feed into the Decision Engine (see Notebook 2).